# 04 · Paneles conservadores y evaluación de estrategias geográficas

**Pregunta.** ¿Cuánta cobertura y qué error fuera de tiempo produce cada estrategia de continuidad territorial?

**Target.** Participación oficial en la elección objetivo: `SV/(SV+NV)`. Una sección con `SV+NV=0` no tiene target evaluable, aunque sí puede recibir predicción.

**Predictor común.** Participación de la elección anterior de la misma familia. Se evalúan 2009→2015, 2012→2018, 2015→2021 y 2018→2024. El fold 2021 es el análogo principal para 2027.

**Escenarios.** (1) sólo clase A; (2) A más componentes B agregados sin repartir conteos; (3) todas las secciones mediante prioridad sección→linaje→municipio→entidad→nacional.

**Interpretación.** Esta evaluación compara estrategias geográficas bajo el mismo predictor transparente. No selecciona todavía el modelo final de 2027 ni estima desempeño futuro garantizado.

**Salidas.** Predicciones fuera de tiempo, métricas, errores por clase, fuentes de fallback, linaje de variables y reporte HTML.

## Contratos y entradas

Se muestran folds, escenarios y recetas de variables antes de calcular predicciones.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs/evaluation.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
required = [
    PROJECT_ROOT / "configs/evaluation.yaml",
    PROJECT_ROOT / "configs/features.yaml",
    PROJECT_ROOT / "artifacts/tables/section_confidence_by_election.parquet",
    PROJECT_ROOT / "artifacts/tables/lineage_components_by_election.parquet",
]
checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required}
display(pd.Series(checks, name="existe").to_frame())
assert all(checks.values())

In [ ]:
with (PROJECT_ROOT / "configs/evaluation.yaml").open(encoding="utf-8") as stream:
    evaluation_config = yaml.safe_load(stream)
with (PROJECT_ROOT / "configs/features.yaml").open(encoding="utf-8") as stream:
    feature_config = yaml.safe_load(stream)

display(pd.DataFrame(evaluation_config["folds"]))
display(pd.DataFrame([
    {"scenario": name, **values}
    for name, values in evaluation_config["scenarios"].items()
]))
display(pd.DataFrame([
    {"variable": name, **values}
    for name, values in feature_config["features"].items()
]))

In [ ]:
display(Markdown(
    "**Interpretación.** Todos los predictores proceden del `feature_year`; el target procede del "
    "`target_year`. La clase geográfica sólo usa metadatos territoriales y eventos AMGE con año "
    "efectivo estrictamente anterior al target."
))

## Ejecución de los tres escenarios

Cada fold se calcula desde conteos persistidos. Los componentes agregan numeradores y denominadores antes de dividir; nunca distribuyen el valor de una sección origen entre descendientes.

In [ ]:
from radar27.modeling.lineage_evaluation import evaluate_scenarios

classes = pd.read_parquet(PROJECT_ROOT / "artifacts/tables/section_confidence_by_election.parquet")
components = pd.read_parquet(PROJECT_ROOT / "artifacts/tables/lineage_components_by_election.parquet")
metrics, predictions, errors_by_class, source_counts = evaluate_scenarios(
    PROJECT_ROOT, classes, components
)
display(metrics[[
    "target_year", "feature_year", "scenario", "target_sections_represented",
    "section_coverage", "nominal_list_coverage", "mae", "rmse", "bias",
    "identified_weighted_mae", "identified_weighted_rmse", "share_within_5pp",
    "evidence_status",
]])

In [ ]:
primary = metrics[metrics.target_year.eq(2021)].set_index("scenario")
stable = primary.loc["stable_only"]
lineage = primary.loc["stable_plus_lineage"]
fallback = primary.loc["hierarchical_fallback"]
display(Markdown(
    f"**Resultado principal 2015→2021.** Sólo A cubre `{stable.section_coverage:.1%}` y logra MAE "
    f"ponderado `{stable.identified_weighted_mae:.3f}`. A+B cubre `{lineage.section_coverage:.1%}` "
    f"con MAE `{lineage.identified_weighted_mae:.3f}`. El respaldo jerárquico cubre "
    f"`{fallback.section_coverage:.1%}` con MAE `{fallback.identified_weighted_mae:.3f}`. "
    "La participación de una elección intermedia no es estacionaria: el sesgo negativo cuantifica "
    "cuánto sobrepredice el rezago de 2015 a 2021."
))

## Cobertura y procedencia de cada predicción

Se cuenta cuántas secciones utilizaron el rezago propio, un componente oficial o un respaldo municipal/estatal. La fuente es un campo por predicción, no una inferencia posterior.

In [ ]:
display(source_counts)
source_totals = source_counts.groupby("target_year", as_index=False).agg(
    total_sections=("sections", "sum"), total_nominal_list=("nominal_list", "sum")
)
source_shares = source_counts.merge(source_totals, on="target_year")
source_shares["section_share"] = source_shares.sections / source_shares.total_sections
source_shares["nominal_list_share"] = source_shares.nominal_list / source_shares.total_nominal_list
display(source_shares)

In [ ]:
primary_sources = source_shares[source_shares.target_year.eq(2021)]
display(Markdown(
    "**Resultado.** En 2021, " + "; ".join(
        f"{row.prediction_source}: {int(row.sections):,} secciones ({row.section_share:.1%})"
        for row in primary_sources.itertuples(index=False)
    ) + ". No fue necesario usar el promedio nacional en ningún fold."
))

## Error por clase territorial

La desagregación detecta si la cobertura adicional se compra a costa de errores concentrados en secciones nuevas o ambiguas.

In [ ]:
display(errors_by_class)

In [ ]:
primary_errors = errors_by_class[errors_by_class.target_year.eq(2021)]
display(Markdown(
    "**Interpretación 2021.** " + "; ".join(
        f"clase {row.geography_confidence_class}: MAE {row.mae:.3f} en {int(row.sections):,} secciones"
        for row in primary_errors.itertuples(index=False)
    ) + ". Estas diferencias justifican conservar la clase como indicador de incertidumbre y "
    "reportar desempeño estratificado."
))

## Linaje de variables y prevención de fuga

El registro siguiente vincula variable, columnas fuente, transformación, disponibilidad, riesgo, política de faltantes e implementación.

In [ ]:
variable_lineage = pd.read_csv(PROJECT_ROOT / "artifacts/tables/variable_lineage.csv")
display(variable_lineage)

leakage_checks = {
    "all_prediction_folds_are_temporal": bool((predictions.feature_year < predictions.target_year).all()),
    "no_missing_predictions": bool(predictions.prediction.notna().all()),
    "predictions_are_probabilities": bool(predictions.prediction.between(0, 1).all()),
    "targets_are_probabilities_when_defined": bool(predictions.actual.dropna().between(0, 1).all()),
}
display(pd.Series(leakage_checks, name="aprobado").to_frame())

In [ ]:
display(Markdown(
    "**Resultado.** Cada predicción conserva `feature_year`, `target_year` y `prediction_source`. "
    "La prueba temporal exige `feature_year < target_year`; ningún promedio se calcula con el "
    "target para producir su propia predicción."
))

## Reporte y decisión

El reporte HTML resume métricas, fuentes y error por clase. La decisión se basa en cobertura, error y auditabilidad, no sólo en la menor cifra.

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/public_lineage_evaluation.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Evaluación de continuidad territorial pública</h1>",
    "<p>Generado por notebooks/04_panel_build_audit.ipynb. El código y outputs completos están en la copia ejecutada.</p>",
    "<h2>Contrato</h2><p>Target SV/(SV+NV); predictor: participación previa de la misma familia.</p>",
    "<h2>Métricas</h2>", metrics.to_html(index=False),
    "<h2>Procedencia de fallback</h2>", source_shares.to_html(index=False),
    "<h2>Error por clase</h2>", errors_by_class.to_html(index=False),
    "<h2>Linaje de variables</h2>", variable_lineage.to_html(index=False),
]), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "artifacts/tables/backtest_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/backtest_metrics.csv",
    PROJECT_ROOT / "artifacts/tables/backtest_errors_by_class.csv",
    PROJECT_ROOT / "artifacts/tables/fallback_source_counts.csv",
    PROJECT_ROOT / "artifacts/tables/variable_lineage.csv",
    report_path,
]
final_checks = {
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    "four_folds_three_scenarios": len(metrics) == 12,
    "hierarchical_defined_target_coverage_is_complete": bool(
        metrics[metrics.scenario.eq("hierarchical_fallback")].section_coverage.eq(1).all()
    ),
    "lineage_improves_2021_coverage": lineage.section_coverage > stable.section_coverage,
    "lineage_does_not_worsen_2021_weighted_mae": (
        lineage.identified_weighted_mae <= stable.identified_weighted_mae
    ),
    "2024_geography_evidence_complete": False,
    **leakage_checks,
}
display(pd.Series(final_checks, name="valor").to_frame())

In [ ]:
display(Markdown(
    f"**Decisión.** Adoptar A+B como muestra principal: aumenta cobertura con error ponderado "
    f"ligeramente menor en el análogo 2021. Usar el escenario jerárquico para cobertura nacional, "
    f"con clase y fuente visibles como incertidumbre. No usar 2024 para validar por sí solo el "
    f"crosswalk porque AMGE termina en 2022. Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))